## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.fundamentals import (
    GROWTH_YEARS,
    MIN_REQUIRED_FIELDS,
    RATING_STRONG_CUTOFF,
    RATING_WEAK_CUTOFF,
    assign_fundamental_rating,
    compute_fundamental_score,
    compute_growth_score,
    compute_health_score,
    compute_quality_score,
    compute_valuation_score,
    growth_from_annual,
)
from src.selection import percentile_score

## 2. Paths & Load Data

Fundamentals only apply to individual stocks -- funds/ETFs report aggregate metrics, not P/E or ROE in a comparable sense, and crypto has none. So this notebook filters the asset universe down to `asset_class == "STOCK"` before doing anything else.

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

asset_universe = pd.read_csv(MANUAL_PATH / "asset_universe_manual.csv")

stock_universe = asset_universe[
    asset_universe["asset_class"] == "STOCK"
].reset_index(drop=True)

stock_universe

## 3. Fetch Fundamentals from yfinance

Eight raw fields, two per scoring dimension (valuation, quality, growth, financial health):

- Valuation, quality and health come from yfinance's current snapshot (`info`). `debtToEquity` is reported in percent there (78 = 0.78x) and is converted to a ratio.
- Growth is the **compound annual growth over the last 3 fiscal years** (revenue and net income, from the annual income statement) — structural growth rather than a single quarter's year-on-year jump, which suits a buy-and-hold view. yfinance only provides ~5 quarters, too few for a trailing-twelve-months comparison. Trade-off: the latest fiscal year can be up to ~12 months old. With fewer years available, the longest span is used (`growth_years`); a loss in the start or end year leaves earnings growth undefined.

Individual fields can be missing -- e.g. `trailingPE` is undefined for a company with negative trailing earnings -- so everything downstream tolerates missing values. A failed download for one ticker only marks that ticker as limited data.

In [ ]:
INFO_FIELDS = {
    "trailingPE": "trailing_pe",
    "priceToBook": "price_to_book",
    "returnOnEquity": "return_on_equity",
    "operatingMargins": "operating_margins",
    "debtToEquity": "debt_to_equity",
    "currentRatio": "current_ratio",
}


def annual_row(statement, label):
    return statement.loc[label] if label in statement.index else pd.Series(dtype=float)


def fetch_fundamentals(ticker):
    row = {"yf_ticker": ticker}
    try:
        stock = yf.Ticker(ticker)
        info = stock.get_info()
        row.update({col: info.get(field) for field, col in INFO_FIELDS.items()})
        if row["debt_to_equity"] is not None:
            row["debt_to_equity"] = row["debt_to_equity"] / 100  # yfinance reports percent

        income = stock.income_stmt
        row["revenue_growth"], revenue_years = growth_from_annual(annual_row(income, "Total Revenue"), GROWTH_YEARS)
        row["earnings_growth"], earnings_years = growth_from_annual(annual_row(income, "Net Income"), GROWTH_YEARS)
        row["growth_years"] = revenue_years or earnings_years
    except Exception as e:
        warnings.warn(f"{ticker}: fundamentals download failed ({e}) -- treated as limited data.")
    return row


fundamentals_raw = pd.DataFrame(
    [fetch_fundamentals(ticker) for ticker in stock_universe["yf_ticker"]]
)

fundamentals_raw

## 4. Merge with Asset Universe

In [ ]:
fundamentals = stock_universe.merge(
    fundamentals_raw,
    on="yf_ticker",
    how="left"
)

RAW_FIELD_COLS = [
    "trailing_pe",
    "price_to_book",
    "return_on_equity",
    "operating_margins",
    "revenue_growth",
    "earnings_growth",
    "debt_to_equity",
    "current_ratio",
]

for col in RAW_FIELD_COLS + ["growth_years"]:
    if col not in fundamentals.columns:
        fundamentals[col] = np.nan
fundamentals[RAW_FIELD_COLS] = fundamentals[RAW_FIELD_COLS].apply(pd.to_numeric, errors="coerce")

fundamentals[["name"] + RAW_FIELD_COLS + ["growth_years"]]

## 5. Data Quality Filter

In [ ]:
fundamentals["n_fields_available"] = fundamentals[RAW_FIELD_COLS].notna().sum(axis=1)

fundamentals["data_quality_ok"] = (
    fundamentals["n_fields_available"] >= MIN_REQUIRED_FIELDS
)

fundamentals[["name", "n_fields_available", "data_quality_ok"]].sort_values(
    "n_fields_available"
)

## 6. Percentile Sub-Scores

Same `percentile_score()` used in Notebook 05 for the return-based selection score -- rank within this universe, invert for "lower is better" metrics (valuation, leverage). Only stocks that pass the data-quality filter are ranked, so a stock with too little data doesn't shift the others' percentiles. With few stocks the ranking is coarse rather than a precise cardinal measure; see the caveat in Section 9.

In [ ]:
SUB_SCORES = {
    # sub-score: (raw field, higher is better)
    "pe_score": ("trailing_pe", False),
    "pb_score": ("price_to_book", False),
    "roe_score": ("return_on_equity", True),
    "operating_margin_score": ("operating_margins", True),
    "revenue_growth_score": ("revenue_growth", True),
    "earnings_growth_score": ("earnings_growth", True),
    "debt_to_equity_score": ("debt_to_equity", False),
    "current_ratio_score": ("current_ratio", True),
}

ranked = fundamentals["data_quality_ok"]
for score_col, (field, higher_is_better) in SUB_SCORES.items():
    fundamentals[score_col] = np.nan
    fundamentals.loc[ranked, score_col] = percentile_score(
        fundamentals.loc[ranked, field], higher_is_better=higher_is_better
    )

fundamentals.head(5)

## 7. Dimension Scores

In [ ]:
fundamentals["valuation_score"] = compute_valuation_score(
    fundamentals["pe_score"], fundamentals["pb_score"]
)
fundamentals["quality_score"] = compute_quality_score(
    fundamentals["roe_score"], fundamentals["operating_margin_score"]
)
fundamentals["growth_score"] = compute_growth_score(
    fundamentals["revenue_growth_score"], fundamentals["earnings_growth_score"]
)
fundamentals["health_score"] = compute_health_score(
    fundamentals["debt_to_equity_score"], fundamentals["current_ratio_score"]
)

fundamentals[
    ["name", "valuation_score", "quality_score", "growth_score", "health_score"]
]

## 8. Final Fundamental Score & Rating

In [ ]:
fundamentals["fundamental_score"] = compute_fundamental_score(
    fundamentals["valuation_score"],
    fundamentals["quality_score"],
    fundamentals["growth_score"],
    fundamentals["health_score"],
)

fundamentals["rating"] = fundamentals.apply(assign_fundamental_rating, axis=1)

fundamentals_ranked = (
    fundamentals
    .sort_values("fundamental_score", ascending=False)
    .reset_index(drop=True)
)

fundamentals_ranked[
    [
        "name",
        "role",
        "source",
        "valuation_score",
        "quality_score",
        "growth_score",
        "health_score",
        "fundamental_score",
        "rating",
    ]
]

## 9. Known Limitation: Relative Percentile Ranking

`percentile_score()` ranks each stock only against the other stocks in your own STOCK-class universe (holdings + watchlist), not against the market. The score is therefore always **relative**: "strong" means strong compared to the other names you track, and adding or removing a stock can shift everyone else's score. How coarse the ranking is depends on the size of the universe -- with only a handful of stocks, one outlier can move another stock's percentile by a full quartile, and an average company can land at the top or bottom simply because the others happen to be weaker or stronger on one metric. The same applies to the return-based ranking in Notebook 05. A broader comparison set (e.g. sector peers or an index) would turn it into a market-relative percentile. Read the score as directional.

The dimension weights (valuation 25 %, quality 30 %, growth 25 %, health 20 %) and rating cutoffs (0.65 / 0.35) are deliberate settings, not calibrated -- see `src/fundamentals.py`.

## 10. Display Table

In [ ]:
fundamentals_display = fundamentals_ranked[
    [
        "yf_ticker",
        "name",
        "role",
        "source",
        "trailing_pe",
        "price_to_book",
        "return_on_equity",
        "operating_margins",
        "revenue_growth",
        "earnings_growth",
        "growth_years",
        "debt_to_equity",
        "current_ratio",
        "valuation_score",
        "quality_score",
        "growth_score",
        "health_score",
        "fundamental_score",
        "data_quality_ok",
        "rating",
    ]
].copy()

fundamentals_display_formatted = fundamentals_display.copy()

score_pct_cols = [
    "valuation_score",
    "quality_score",
    "growth_score",
    "health_score",
    "fundamental_score",
]

fundamentals_display_formatted[score_pct_cols] = (
    fundamentals_display_formatted[score_pct_cols] * 100
)

fundamentals_display_formatted

## 11. Visualisation

In [ ]:
plot_df = fundamentals_ranked.dropna(subset=["fundamental_score"]).copy()

plt.figure(figsize=(10, 5))

plt.bar(
    plot_df["name"],
    plot_df["fundamental_score"] * 100
)

plt.title("Fundamental Score by Stock")
plt.xlabel("Stock")
plt.ylabel("Fundamental Score (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    plot_df["valuation_score"] * 100,
    plot_df["quality_score"] * 100,
    s=plot_df["growth_score"] * 400 + 40
)

for _, row in plot_df.iterrows():
    plt.annotate(
        row["name"],
        (row["valuation_score"] * 100, row["quality_score"] * 100),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=9
    )

plt.title("Valuation vs. Quality (bubble size = Growth Score)")
plt.xlabel("Valuation Score (%, higher = cheaper)")
plt.ylabel("Quality Score (%)")
plt.tight_layout()
plt.show()

## 12. Rating Summary

In [ ]:
rating_summary = (
    fundamentals_ranked
    .groupby("rating")
    .size()
    .reset_index(name="n_stocks")
    .sort_values("n_stocks", ascending=False)
)

rating_summary

## 13. Exports

In [ ]:
fundamentals_ranked.to_csv(
    DATA_PATH / "fundamental_scoring.csv",
    index=False
)

fundamentals_display_formatted.to_csv(
    DATA_PATH / "fundamental_scoring_display.csv",
    index=False
)

print("Fundamental scoring exports saved.")